In [1]:
import ee
import geemap
import os
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Load Configuration
import yaml
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Ensure data directories exist
raw_img_path = "../data/raw/satellite_images"
raw_tab_path = "../data/raw/tabular"
os.makedirs(raw_img_path, exist_ok=True)
os.makedirs(raw_tab_path, exist_ok=True)

print("Libraries imported and directories verified.")

Libraries imported and directories verified.


In [ ]:
# Initialize Google Earth Engine
try:
    ee.Initialize()
    print("Google Earth Engine initialized successfully!")
except Exception as e:
    print("Authentication required. Follow the steps below:")
    ee.Authenticate()
    ee.Initialize()
    print("Google Earth Engine initialized successfully after authentication!")

In [ ]:
# Define coordinates for Austin, Texas
city_center = ee.Geometry.Point([-97.7431, 30.2672])

# REDUCED BUFFER: 5000m (5km) instead of 10000m to fit download limits
roi = city_center.buffer(5000).bounds() 

Map = geemap.Map(center=[30.2672, -97.7431], zoom=13)
Map.addLayer(roi, {'color': 'red'}, "Study Area")
Map

In [ ]:
def get_sentinel_image(roi, start_date, end_date):
    # Load Sentinel-2 Surface Reflectance
    s2 = ee.ImageCollection('COPERNICUS/S2_SR') \
        .filterBounds(roi) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
        .median() \
        .clip(roi)
    
    # Select RGB and Near-Infrared bands (B4, B3, B2, B8)
    return s2.select(['B4', 'B3', 'B2', 'B8'])

# Fetch Data
start_date = '2023-06-01'
end_date = '2023-08-31'
satellite_img = get_sentinel_image(roi, start_date, end_date)

print(f"Sentinel-2 Data Fetched for {start_date} to {end_date}")

# Visualization parameters
vis_params = {
    'min': 0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'] # RGB
}
Map.addLayer(satellite_img, vis_params, "Sentinel-2 RGB")
Map

In [ ]:
filename = os.path.join(raw_img_path, "austin_sentinel_2023.tif")

print("Downloading satellite imagery (5km radius)...")
try:
    geemap.ee_export_image(
        satellite_img,
        filename=filename,
        scale=10, 
        region=roi,
        file_per_band=False
    )
    
    # Verify file existence and size
    if os.path.exists(filename):
        size_mb = os.path.getsize(filename) / (1024 * 1024)
        print(f"Success! Saved to: {filename}")
        print(f"File Size: {size_mb:.2f} MB")
    else:
        print("Error: File was not created.")
        
except Exception as e:
    print(f"Error during download: {e}")

In [11]:
# Ticker: XLU (Utilities), XLE (Energy), CL=F (Crude Oil Futures)
tickers = ['XLU', 'XLE', 'CL=F']

print(f"Fetching market data for: {tickers}...")
market_data = yf.download(tickers, start="2023-01-01", end="2023-12-31")

# Clean and flatten columns if necessary (handling MultiIndex)
market_data.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in market_data.columns.values]

# Save to CSV
csv_path = os.path.join(raw_tab_path, "energy_market_data.csv")
market_data.to_csv(csv_path)

print(f"Market data saved to: {csv_path}")
print(market_data.head())

[                       0%                       ]

Fetching market data for: ['XLU', 'XLE', 'CL=F']...


[*********************100%***********************]  3 of 3 completed

Market data saved to: ../data/raw/tabular\energy_market_data.csv
            Close_CL=F  Close_XLE  Close_XLU  High_CL=F   High_XLE   High_XLU  \
Date                                                                            
2023-01-03   76.930000  38.125160  32.144360  81.500000  39.462251  32.344978   
2023-01-04   72.839996  38.120640  32.436172  77.419998  38.341982  32.727979   
2023-01-05   73.669998  38.816292  31.734005  74.919998  39.028601  32.313061   
2023-01-06   73.769997  39.552586  32.372334  75.470001  40.035926  32.527354   
2023-01-09   74.629997  39.412560  32.591194  76.739998  40.212102  32.837403   

             Low_CL=F    Low_XLE    Low_XLU  Open_CL=F   Open_XLE   Open_XLU  \
Date                                                                           
2023-01-03  76.599998  37.650853  31.706650  80.570000  39.263493  32.235548   
2023-01-04  72.730003  37.415958  32.240115  77.250000  37.560508  32.290269   
2023-01-05  72.459999  37.948990  31.638256  73

In [12]:
# Simulation: Create a grid of points over the ROI
# In a real scenario, this would be joined with Census Tract Shapefiles
num_samples = 500

# Random coordinates within the Austin area bounding box
lat_min, lat_max = 30.15, 30.35
lon_min, lon_max = -97.85, -97.65

data = {
    'latitude': np.random.uniform(lat_min, lat_max, num_samples),
    'longitude': np.random.uniform(lon_min, lon_max, num_samples),
    
    # Synthetic Features based on real-world correlations
    'median_income': np.random.normal(75000, 25000, num_samples), # Avg income
    'population_density': np.random.randint(1000, 8000, num_samples), # People per sq mile
    'avg_building_age': np.random.randint(5, 80, num_samples), # Age of buildings
    
    # Target Variable: Energy Consumption Index (simulated)
    # Higher density + Older buildings = Higher Consumption
    'energy_consumption_kwh': np.zeros(num_samples) 
}

df_socio = pd.DataFrame(data)

# Introduce logic to target variable to make it learnable
df_socio['energy_consumption_kwh'] = (
    (df_socio['population_density'] * 0.5) + 
    (df_socio['avg_building_age'] * 20) + 
    np.random.normal(0, 500, num_samples)
)

# Save
socio_path = os.path.join(raw_tab_path, "socio_economic_data.csv")
df_socio.to_csv(socio_path, index=False)
print(f"Socio-economic data generated and saved to: {socio_path}")

Socio-economic data generated and saved to: ../data/raw/tabular\socio_economic_data.csv
